In [ ]:
# setup (instancia con jaccard+top_k, evaluador)
import numpy as np, pandas as pd
from abs_affinity_based_slotting.config import RAW_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.demand import build_cooccurrence, build_sku_demand, affinity_registry, filter_registry
from abs_affinity_based_slotting.warehouse import occupied_locations, build_location_costs, build_bay_distance_matrix
from abs_affinity_based_slotting.slotting import build_instance
from abs_affinity_based_slotting.clustering import clustering_registry
from abs_affinity_based_slotting.methods import (CurrentSlotting, DemandGreedySlotting,
    LinearAssignmentSlotting, SwapSearchSlotting, BiLevelSlotting)
from abs_affinity_based_slotting.evaluation import Evaluator

ds = WarehouseDataLoader(RAW_DIR).load_all()
split = split_picking_events(ds.picking_events, test_size=0.2)
universe = occupied_locations(ds.initial_stock)["sku"].to_numpy()
co = build_cooccurrence(split.train, skus=universe)
A = filter_registry.get("top_k")(k=10).filter(affinity_registry.get("jaccard")().build(co.matrix, co.support, co.n_batches))
instance = build_instance(build_sku_demand(split.train), build_location_costs(ds.initial_stock, ds.distances),
                          build_bay_distance_matrix(ds.distances), initial_stock=ds.initial_stock, skus=universe, affinity=A)
evaluator = Evaluator.from_tables(ds.coordinates, ds.distances, ds.initial_stock)

In [ ]:
# posicion en el recorrido (snake) para el Problema 1 compacto
coord = ds.coordinates.set_index("bay_id")
bay_key = coord["aisle"].astype(float) * 1000.0 + coord["bay_number"].astype(float)
bay_key = bay_key.reindex(instance.bay_ids).to_numpy()
bay_key = np.nan_to_num(bay_key, nan=np.nanmax(bay_key) + 1.0)
snake_pos = bay_key[instance.location_bay]

In [ ]:
# helper
def run(name, method):
    sol = method.solve(instance)
    m = evaluator.evaluate(sol, split.test)
    return {"metodo": name, "mean": round(m.mean_batch_distance), "p95": round(m.p95_batch_distance)}

In [ ]:
# pregunta: la afinidad dentro de la zona de vendor mejora las rutas?
# Problema 1 fijo y compacto (snake); se varia el solver de zona (Problema 2).
clu = clustering_registry.get("merchant")()
rows = [
    run("demand_greedy (ref)", DemandGreedySlotting()),
    run("bilevel snake, zona=linear (lam=1)", BiLevelSlotting(clu, LinearAssignmentSlotting(), location_cost=snake_pos)),
    run("bilevel snake, zona=swaps lam=0.7",  BiLevelSlotting(clu, SwapSearchSlotting(lam=0.7), location_cost=snake_pos)),
    run("bilevel snake, zona=swaps lam=0.5",  BiLevelSlotting(clu, SwapSearchSlotting(lam=0.5), location_cost=snake_pos)),
    run("bilevel snake, zona=swaps lam=0.3",  BiLevelSlotting(clu, SwapSearchSlotting(lam=0.3), location_cost=snake_pos)),
]
pd.DataFrame(rows)